In [1]:
image_path="grey_1600x1300.jpg"


In [2]:

from PIL import Image
import numpy as np

import numpy as np
from PIL import Image

def image_to_tensor(filepath, dtype=np.float32, normalize=True, grayscale=False):
    """
    Convert an image to tensor format (1, C, H, W).

    Parameters
    ----------
    filepath : str
        Path to image.
    dtype : numpy dtype
        Output dtype (e.g., np.float32 or np.int32).
    normalize : bool
        If True and dtype is float32, scale to [0, 1].
    grayscale : bool
        If True -> C=1, else C=3 (RGB).

    Returns
    -------
    img_tensor : np.ndarray
        Shape (1, C, H, W)
    """

    # Select mode
    mode = 'L' if grayscale else 'RGB'

    # Open and convert image
    image = Image.open(filepath).convert(mode)

    # Convert to NumPy array
    img_np = np.array(image, dtype=dtype)

    # Normalize or clip
    if dtype == np.float32:
        if normalize:
            img_np /= 255.0
    elif dtype == np.int32:
        img_np = np.clip(img_np, 0, 255)

    # Ensure channel dimension exists
    if grayscale:
        # (H, W) -> (1, H, W)
        img_np = np.expand_dims(img_np, axis=0)
    else:
        # (H, W, 3) -> (3, H, W)
        img_np = np.transpose(img_np, (2, 0, 1))

    # Add batch dimension -> (1, C, H, W)
    img_tensor = np.expand_dims(img_np, axis=0)

    return img_tensor
    
def tensor_to_image(img_tensor, filepath, denormalize=True):
    img_np = np.squeeze(img_tensor, axis=0)
    num_channels = img_np.shape[0]
    
    if num_channels == 1:
        img_np = np.squeeze(img_np, axis=0)
        mode = 'L'
    elif num_channels == 3:
        img_np = np.transpose(img_np, (1, 2, 0))
        mode = 'RGB'
    else:
        raise ValueError(f"Unsupported number of channels: {num_channels}")
    
    if img_np.dtype in [np.float32, np.float64]:
        if denormalize:
            img_np = img_np * 255.0
        img_np = np.clip(img_np, 0, 255).astype(np.uint8)
    else:
        img_np = np.clip(img_np, 0, 255).astype(np.uint8)
    
    image = Image.fromarray(img_np, mode=mode)
    image.save(filepath)
    print(f"Image saved to: {filepath}")

In [3]:

def create_kernel(kernel_size=5):

    channels = 1

    # Create empty kernel: shape [1, channels, kernel_size, kernel_size]
    mean_kernel = np.zeros((1, channels, kernel_size, kernel_size), dtype=np.float32)

    for c in range(channels):
        mean_kernel[0, c, :, :] = 1. / (kernel_size ** 2)

    mean_kernel_q =  pow(2, 8)*mean_kernel
    mean_kernel_q = mean_kernel_q.astype(np.int32)

    return mean_kernel,mean_kernel_q

import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image
import torch.nn.functional as F

def exec_conv2d(np_x,np_w,stride=5,padding=0):
    torch_type= torch.float32
    match np_x.dtype:            
        case np.int32:
            torch_type = torch.int32
    torch_x = torch.from_numpy(np_x).to(torch_type)
    torch_w = torch.from_numpy(np_w).to(torch_type)
#Apply convolution
    output = F.conv2d(
        torch_x,
        torch_w,
        bias=None,
        stride=stride,
        padding=padding
    )
    return output

def serialize(y,x,w,sufix=""):
    np.save(f"y{sufix}",y)
    np.save(f"x{sufix}",x)
    np.save(f"w{sufix}",w)


In [4]:
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image
import torch.nn.functional as F

kernel_size=5
stride=5
grayscale=True
mean_kernel,mean_kernel_q = create_kernel(kernel_size=kernel_size)
print(f"kernel size:{mean_kernel_q.shape}")
np_x= image_to_tensor(image_path,grayscale=grayscale)
print(f"np_x.shape={np_x.shape}")
np_x_q= image_to_tensor(image_path,dtype=np.int32,grayscale=grayscale)
output = exec_conv2d(np_x,mean_kernel,stride=stride)
print(f"float shape{output.shape}")
#output =output.clamp(0, 1)
serialize(y=output.numpy(),x=np_x,w=mean_kernel)
#save jpg
output = T.ToPILImage()(output.squeeze(0))
output.save("output_downsampled.jpg")

#
output = exec_conv2d(np_x_q,mean_kernel_q,stride=stride)
#de-scale the kernel values
output =output/(pow(2, 8))
output = output.to(torch.int32)
print(f"int32 size:{output.shape}")
serialize(y=output.numpy(),x=np_x_q,w=mean_kernel_q,sufix="_int32")
np_x_q.tofile("x_int32.bin")
#save jpg
output = T.ToPILImage()(output.squeeze(0).to(torch.uint8))
output.save("output_downsampled_q.jpg")


kernel size:(1, 1, 5, 5)
np_x.shape=(1, 1, 1300, 1600)
float shapetorch.Size([1, 1, 260, 320])
int32 size:torch.Size([1, 1, 260, 320])


In [5]:
print(mean_kernel)

[[[[0.04 0.04 0.04 0.04 0.04]
   [0.04 0.04 0.04 0.04 0.04]
   [0.04 0.04 0.04 0.04 0.04]
   [0.04 0.04 0.04 0.04 0.04]
   [0.04 0.04 0.04 0.04 0.04]]]]


In [8]:
kernel_flat=mean_kernel_q.flatten().astype(np.uint32)
print(",".join(str(val) for val in kernel_flat))

10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10
